In [1]:
# !pip install -q pandas numpy scikit-learn matplotlib

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.calibration import CalibratedClassifierCV
from joblib import dump

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [3]:
# Caminho do CSV salvo no notebook anterior
arquivo = "dataset_unificado.csv"

# Carregar o dataset
df = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")

# Mantém apenas as colunas necessárias
df = df[['texto', 'label']].dropna(subset=['texto', 'label']).copy()

# Garante binário inteiro (0/1)
df['label'] = df['label'].astype(int)

# (Opcional) limpeza super leve (sem remover acentos/pontuação agressivamente)
def pre_clean(s):
    s = str(s)
    s = s.replace("\n", " ").replace("\r", " ")
    return " ".join(s.split())

df['texto'] = df['texto'].map(pre_clean)

df.head(), df['label'].value_counts(normalize=True).round(3)


(                                               texto  label
 0  Ancine aprovou em 2019 a captação de 530 mil r...      0
 1  Parte dos trechos das gravações utilizadas no ...      0
 2  Em uma publicação no Twitter, o deputado usa c...      0
 3  Procurada pelo Comprova, médica enviou 34 estu...      0
 4  Ao contrário do que afirma uma médica em vídeo...      0,
 label
 0    0.679
 1    0.321
 Name: proportion, dtype: float64)

In [4]:
X = df['texto'].values
y = df['label'].values

# 80/20 primeiro
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Depois, separa validação a partir do train_val → 12.5% do total ≈ 10%/10%/80%
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=RANDOM_STATE, stratify=y_train_val
)

len(X_train), len(X_val), len(X_test)


(21509, 3073, 6146)

In [5]:
svm_pipe = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(
        # mistura sinais de palavras e um pouco de contexto
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=True,
        strip_accents=None,   # mantenho acentos porque português
        analyzer="word"
    )),
    ("clf", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))
])


In [7]:
logreg_base = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=True,
        analyzer="word"
    )),
    ("clf", LogisticRegression(
        max_iter=1000, n_jobs=None, class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

# Calibra com sigmoid para ter probas melhores (demora um pouco mais)
logreg_pipe = CalibratedClassifierCV(
    estimator=logreg_base, method="sigmoid", cv=5
)



In [8]:
def fit_and_eval(pipe, name="model"):
    pipe.fit(X_train, y_train)

    def eval_on(split_name, Xs, ys):
        y_pred = pipe.predict(Xs)
        acc = accuracy_score(ys, y_pred)
        p, r, f1, _ = precision_recall_fscore_support(ys, y_pred, average="binary", zero_division=0)
        print(f"[{name}] {split_name} -> acc: {acc:.3f} | P: {p:.3f} | R: {r:.3f} | F1: {f1:.3f}")
        print(confusion_matrix(ys, y_pred))
        print(classification_report(ys, y_pred, zero_division=0))
        return acc, p, r, f1

    print("=== VALID ===")
    eval_on("valid", X_val, y_val)
    print("\n=== TEST ===")
    acc, p, r, f1 = eval_on("test ", X_test, y_test)

    # AUC só se o modelo tiver decision_function ou predict_proba
    try:
        if hasattr(pipe, "decision_function"):
            scores = pipe.decision_function(X_test)
        elif hasattr(pipe, "predict_proba"):
            scores = pipe.predict_proba(X_test)[:, 1]
        else:
            scores = None
        if scores is not None:
            auc = roc_auc_score(y_test, scores)
            print(f"AUC: {auc:.3f}")
    except Exception as e:
        print("AUC não calculada:", e)

    return pipe

# 5.1 Treinar SVM
svm_trained = fit_and_eval(svm_pipe, name="LinearSVC")

# 5.2 (Opcional) Treinar LogReg calibrada
# logreg_trained = fit_and_eval(logreg_pipe, name="LogRegCalibrated")


=== VALID ===
[LinearSVC] valid -> acc: 0.941 | P: 0.897 | R: 0.921 | F1: 0.909
[[1984  104]
 [  78  907]]
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      2088
           1       0.90      0.92      0.91       985

    accuracy                           0.94      3073
   macro avg       0.93      0.94      0.93      3073
weighted avg       0.94      0.94      0.94      3073


=== TEST ===
[LinearSVC] test  -> acc: 0.953 | P: 0.911 | R: 0.946 | F1: 0.928
[[3993  182]
 [ 106 1865]]
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      4175
           1       0.91      0.95      0.93      1971

    accuracy                           0.95      6146
   macro avg       0.94      0.95      0.95      6146
weighted avg       0.95      0.95      0.95      6146

AUC: 0.985


In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(svm_pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
print("CV F1 (5 folds):", np.round(cv_scores, 3), " | mean:", cv_scores.mean().round(3))


CV F1 (5 folds): [0.925 0.932 0.928 0.929 0.926]  | mean: 0.928


In [10]:
# Salva o pipeline inteiro (TF-IDF + modelo) para uso direto
os.makedirs("models", exist_ok=True)
dump(svm_trained, "models/svm_tfidf.joblib")
print("Modelo salvo em: models/svm_tfidf.joblib")


Modelo salvo em: models/svm_tfidf.joblib


In [11]:
from joblib import load

pipe = load("models/svm_tfidf.joblib")

novos_textos = [
    "Ministério confirma que isso aconteceu e pede que compartilhe...",
    "Corrente de WhatsApp diz que 'fulano' fez tal coisa impossível..."
]

pred = pipe.predict(novos_textos)
for t, yhat in zip(novos_textos, pred):
    classe = "VERÍDICO" if yhat == 0 else "FALSO"
    print(classe, "→", t[:100], "...")


FALSO → Ministério confirma que isso aconteceu e pede que compartilhe... ...
FALSO → Corrente de WhatsApp diz que 'fulano' fez tal coisa impossível... ...


In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

# 1) Filtrar fontes e remover textos inválidos (NaN ou só espaços)
df_filtrado = (
    df[df['fonte'] != 'fakenews_multilabel']
      .copy()
)
df_filtrado = df_filtrado[df_filtrado['texto'].notna()]
df_filtrado = df_filtrado[df_filtrado['texto'].astype(str).str.strip().ne('')]

# 2) Separar X e y
X = df_filtrado['texto'].astype(str)
y = df_filtrado['label'].astype(int)

# 3) Predições com o pipeline já treinado/carregado
y_pred = pipe.predict(X)

# 4) Montar tabela por fonte
linhas = []
for fonte in df_filtrado['fonte'].unique():
    mask = (df_filtrado['fonte'] == fonte)
    y_true_f = y[mask]
    y_pred_f = y_pred[mask]

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_f, y_pred_f, labels=[0, 1], zero_division=0
    )

    # 0 = VERÍDICO, 1 = FALSO (conforme nosso mapeamento)
    linhas.append([fonte, "VERÍDICO", prec[0], rec[0], f1[0]])
    linhas.append([fonte, "FALSO",    prec[1], rec[1], f1[1]])
    linhas.append([fonte, "Average",  np.mean(prec), np.mean(rec), np.mean(f1)])

# 5) DataFrame final, ordenado
tabela = pd.DataFrame(linhas, columns=["fonte", "class", "precision", "recall", "F1"])

ordem_classe = {"VERÍDICO": 0, "FALSO": 1, "Average": 2}
tabela['ordem'] = tabela['class'].map(ordem_classe)
tabela = tabela.sort_values(by=["fonte", "ordem"]).drop(columns="ordem").reset_index(drop=True)

# 6) Exibir bonito
pd.set_option("display.float_format", "{:.6f}".format)
print(tabela)


              fonte     class  precision   recall       F1
0  central_de_fatos  VERÍDICO   0.992451 0.997002 0.994721
1  central_de_fatos     FALSO   0.864542 0.716172 0.783394
2  central_de_fatos   Average   0.928496 0.856587 0.889057
3      fake_recogna  VERÍDICO   0.987089 0.966133 0.976499
4      fake_recogna     FALSO   0.966924 0.987397 0.977054
5      fake_recogna   Average   0.977007 0.976765 0.976776
6       fakebr_nilc  VERÍDICO   0.992688 0.980828 0.986723
7       fakebr_nilc     FALSO   0.981060 0.992778 0.986884
8       fakebr_nilc   Average   0.986874 0.986803 0.986803
